In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from torch import nn
import numpy as np
import copy
import matplotlib.pyplot as plt
from collections import Counter

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32
hidden_units = 64
epochs = 30
epochs_1 = 25
epochs_2 = 10
lr=0.005
lr_1 = lr * 0.1
lr_2 = lr_1 * 0.1
weight_decay= 0.9e-2

In [ ]:
train_root = "/kaggle/input/catdog-combined-dataset-hust2024/CatDog/CatDog/training_set/training_set"
test_root = '/kaggle/input/catdog-combined-dataset-hust2024/CatDog/CatDog/test_set/test_set'

In [ ]:
image_transforms = {
    'train': transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        #transforms.RandomRotation(360), # random rotation
        transforms.RandomHorizontalFlip(), # random horizontal flip
        #transforms.RandomVerticalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]) # Normalize to [-1, 1]
    ]),
    'transform1': transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomRotation(degrees=(90,90)),
        #transforms.RandomHorizontalFlip(), # random horizontal flip
        transforms.RandomVerticalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]) # Normalize to [-1, 1]
    ]),
    'transform2': transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomRotation(degrees=(180,180)),
        transforms.RandomHorizontalFlip(), # random horizontal flip
        #transforms.RandomVerticalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]) # Normalize to [-1, 1]
    ]),
    'transform3': transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomRotation(degrees=(270,270)),
        #transforms.RandomHorizontalFlip(), # random horizontal flip
        transforms.RandomVerticalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]) # Normalize to [-1, 1]
    ]),
    'test': transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
    ])
}

original_train_dataset = datasets.ImageFolder(root=train_root, transform=image_transforms['train'])
add_train_dataset_1 = datasets.ImageFolder(root=train_root, transform=image_transforms['transform1'])
add_train_dataset_2 = datasets.ImageFolder(root=train_root, transform=image_transforms['transform2'])
add_train_dataset_3 = datasets.ImageFolder(root=train_root, transform=image_transforms['transform3'])

In [ ]:
train_dataset = torch.utils.data.ConcatDataset([original_train_dataset,add_train_dataset_1,add_train_dataset_2,add_train_dataset_3])
test_dataset = datasets.ImageFolder(root=test_root, transform=image_transforms['test'])

In [ ]:
class_to_index = original_train_dataset.class_to_idx
print("Class to index mapping:")
print(class_to_index)

In [ ]:
class_names = original_train_dataset.classes
print(class_names)

In [ ]:
fig = plt.figure(figsize=(9, 9))
rows, cols = 3, 3
for i in range(1, rows * cols + 1):
    random_idx = torch.randint(0, len(train_dataset), size=[1]).item()
    img, _ = train_dataset[random_idx]

    for dataset in train_dataset.datasets:
        length = len(dataset.targets)
        if random_idx < length:
            break
        random_idx -= length
    target = dataset.targets[random_idx]
    mean = torch.tensor([0.5, 0.5, 0.5])
    std = torch.tensor([0.5, 0.5, 0.5])
    img = img * std[:, None, None] + mean[:, None, None]  
    img = img.permute(1, 2, 0).numpy()

    fig.add_subplot(rows, cols, i)
    plt.imshow(img)
    plt.title(class_names[target])
    plt.axis(False)

plt.show()


In [ ]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,num_workers=4)
len(train_loader),len(test_loader)

In [ ]:
def accuracy(y_true, y_pred):
    correct = (y_true == y_pred).sum().item()
    total = y_true.size(0)
    return 100*correct / total

In [ ]:
def train_step(model,
               data_loader,
               loss_fn,
               optimizer,
               accuracy_fn = accuracy,
               device = device):
    train_loss,train_acc =0,0
    model.train()
    for batch, (X,y) in enumerate(data_loader):
        X,y = X.to(device), y.to(device).type(torch.float)
        y_logits = model(X).squeeze()
        y_pred = torch.round(torch.sigmoid(y_logits))
        loss = loss_fn(y_logits,y)
        train_loss += loss.item() * X.size(0) 
        train_acc += accuracy_fn(y,y_pred) * X.size(0) 
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    train_loss /= len(data_loader.dataset)
    train_acc /= len(data_loader.dataset)
    print("Train loss:",train_loss,", Train acc:",train_acc)
    return train_acc

In [ ]:
def test_step(model,
              data_loader,
              loss_fn,
              accuracy_fn = accuracy,
              device = device):
    test_loss,test_acc =0,0
    model.eval()
    with torch.inference_mode():
        for X,y in data_loader:
            X,y = X.to(device), y.to(device).type(torch.float)
            test_logits = model(X).squeeze()
            test_pred = torch.round(torch.sigmoid(test_logits)).squeeze()
            test_loss += loss_fn(test_logits,y) * X.size(0) 
            test_acc += accuracy_fn(y, test_pred) * X.size(0) 
        test_loss /= len(data_loader.dataset)
        test_acc /= len(data_loader.dataset)
    print("Test loss:",test_loss.item(),", Test acc:",test_acc)
    return test_acc


In [ ]:
class CatDogModel(nn.Module):
    def __init__(self,input_shape, hidden_units, output_shape,dropout_prob = 0.2):
        super().__init__()
        self.conv_block_1 = nn.Sequential(
            nn.Conv2d(in_channels=input_shape,
                      out_channels = hidden_units,
                      kernel_size = 3,
                      stride = 1,
                      padding = 1),
            nn.ReLU(),
            nn.BatchNorm2d(hidden_units),
            nn.Conv2d(in_channels = hidden_units,
                      out_channels = hidden_units,
                      kernel_size = 3,
                      stride = 1,
                      padding = 1),
            nn.ReLU(),
            nn.BatchNorm2d(hidden_units),
            nn.MaxPool2d(kernel_size = 2)
        )
        self.conv_block_2 = nn.Sequential(
            nn.Conv2d(in_channels=hidden_units,
                      out_channels = hidden_units*2,
                      kernel_size = 3,
                      stride = 1,
                      padding = 1),
            nn.ReLU(),
            nn.BatchNorm2d(hidden_units*2),
            nn.Conv2d(in_channels = hidden_units*2,
                      out_channels = hidden_units*2,
                      kernel_size = 3,
                      stride = 1,
                      padding = 1),
            nn.ReLU(),
            nn.BatchNorm2d(hidden_units*2),
            nn.MaxPool2d(kernel_size = 2)
        )
        self.conv_block_3 = nn.Sequential(
            nn.Conv2d(in_channels=hidden_units*2,
                      out_channels = hidden_units*4,
                      kernel_size = 3,
                      stride = 1,
                      padding = 1),
            nn.ReLU(),
            nn.BatchNorm2d(hidden_units*4),
            nn.Conv2d(in_channels = hidden_units*4,
                      out_channels = hidden_units*4,
                      kernel_size = 3,
                      stride = 1,
                      padding = 1),
            nn.ReLU(),
            nn.BatchNorm2d(hidden_units*4),
            nn.Conv2d(in_channels = hidden_units*4,
                      out_channels = hidden_units*4,
                      kernel_size = 3,
                      stride = 1,
                      padding = 1),
            nn.ReLU(),
            nn.BatchNorm2d(hidden_units*4),
            nn.MaxPool2d(kernel_size = 2)
        )

        self.conv_block_4 = nn.Sequential(
            nn.Conv2d(in_channels=hidden_units*4,
                      out_channels = hidden_units*8,
                      kernel_size = 3,
                      stride = 1,
                      padding = 1),
            nn.ReLU(),
            nn.BatchNorm2d(hidden_units*8),
            nn.Conv2d(in_channels = hidden_units*8,
                      out_channels = hidden_units*8,
                      kernel_size = 3,
                      stride = 1,
                      padding = 1),
            nn.ReLU(),
            nn.BatchNorm2d(hidden_units*8),
            nn.Conv2d(in_channels = hidden_units*8,
                      out_channels = hidden_units*8,
                      kernel_size = 3,
                      stride = 1,
                      padding = 1),
            nn.ReLU(),
            nn.BatchNorm2d(hidden_units*8),
            nn.MaxPool2d(kernel_size = 2)
        )

        self.conv_block_5 = nn.Sequential(
            nn.Conv2d(in_channels=hidden_units*8,
                      out_channels = hidden_units*8,
                      kernel_size = 3,
                      stride = 1,
                      padding = 1),
            nn.ReLU(),
            nn.BatchNorm2d(hidden_units*8),
            nn.Conv2d(in_channels = hidden_units*8,
                      out_channels = hidden_units*8,
                      kernel_size = 3,
                      stride = 1,
                      padding = 1),
            nn.ReLU(),
            nn.BatchNorm2d(hidden_units*8),
            nn.Conv2d(in_channels = hidden_units*8,
                      out_channels = hidden_units*8,
                      kernel_size = 3,
                      stride = 1,
                      padding = 1),
            nn.ReLU(),
            nn.BatchNorm2d(hidden_units*8),
            nn.MaxPool2d(kernel_size = 2)
        )
        flatten_size = self.initialize_classifier(input_shape)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout_prob),
            nn.Linear(in_features=flatten_size,
                      out_features=4096),
            nn.ReLU(),
            nn.Dropout(dropout_prob),
            nn.Linear(in_features=4096,
                      out_features=4096),
            nn.ReLU(),
            nn.Dropout(dropout_prob),
            nn.Linear(in_features=4096,
                      out_features=output_shape),
        )

    def initialize_classifier(self, input_shape):
        dummy_input = torch.zeros(1, input_shape, IMG_SIZE, IMG_SIZE)
        dummy_output = self.conv_block_1(dummy_input)
        dummy_output = self.conv_block_2(dummy_output)
        dummy_output = self.conv_block_3(dummy_output)
        dummy_output = self.conv_block_4(dummy_output)
        dummy_output = self.conv_block_5(dummy_output)
        flatten_size = dummy_output.numel()
        print(flatten_size)
        return flatten_size

    def forward(self,x):
        x = self.conv_block_1(x)
        x= self.conv_block_2(x)
        x = self.conv_block_3(x)
        x = self.conv_block_4(x)
        x = self.conv_block_5(x)
        x = self.classifier(x)
        return x

In [ ]:
model = CatDogModel(input_shape=3,
                        hidden_units=hidden_units,
                        output_shape=1,
                    dropout_prob = 0.5
                    ).to(device)

In [ ]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params}")

# Trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable_params}")

# Non-trainable parameters
non_trainable_params = total_params - trainable_params
print(f"Non-trainable parameters: {non_trainable_params}")

In [ ]:
def train_model(epochs,model, train_loader = train_loader,test_loader = test_loader,lr=lr,weight_decay=weight_decay ,accuracy_fn=accuracy,device=device):
    loss_fn = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr,weight_decay=weight_decay)
    best_model = None
    train_acc_list = []
    test_acc_list = []
    best_acc = 0
    for epoch in range(epochs):
        print("Epoch:",epoch+1)
        train_acc = train_step(model= model,
                   data_loader= train_loader,
                   loss_fn= loss_fn,
                   optimizer=optimizer,
                   accuracy_fn=accuracy,
                   device=device)
        test_acc = test_step(model=model,
                  data_loader=test_loader,
                  loss_fn=loss_fn,
                  accuracy_fn=accuracy,
                  device=device)
        train_acc_list.append(train_acc)
        test_acc_list.append(test_acc)
        if test_acc > best_acc:
            best_acc = test_acc
            best_model = copy.deepcopy(model)
        print("Best Acc:",best_acc)
    torch.save(best_model.state_dict(), "CatDog_model_with_%.3f%%.pth" % (max(test_acc_list)))
    plt.plot(range(1, epochs + 1), np.array(train_acc_list), color = "r", label = "Train accuracy")
    plt.plot(range(1, epochs + 1), np.array(test_acc_list), color = "b", label = "Validation accuracy")
    plt.xlabel("Epochs")
    plt.ylabel("Accuracy(%)")
    plt.title("Train accuracy vs Test accuracy")
    plt.legend()
    plt.show()
    return best_model,best_acc

In [ ]:
best_model,best_acc = train_model(epochs,model,lr = lr)

In [ ]:
best_model_copy = copy.deepcopy(best_model)
best_model_1,best_acc_1 = train_model(epochs_1,best_model_copy,lr = lr_1)

In [ ]:
if best_acc_1 > best_acc:
    best_model_1_copy = copy.deepcopy(best_model_1)
else:
    best_model_1_copy = copy.deepcopy(best_model)
best_model_2,best_acc_2 = train_model(epochs_2,best_model_1_copy,lr = lr_2)

In [ ]:
print(f"Best Accuracy after first training: {best_acc:.3f}%")
print(f"Best Accuracy after second training: {best_acc_1:.3f}%")
print(f"Best Accuracy after final training: {best_acc_2:.3f}%")